<div style="text-align: center; font-weight: bold;">
    <h1>EHR Preprocessing 1: Data Cleaning</h1>
    <h4>Author: Vidul Ayakulangara Panickan</h4>
</div>

**Goal:** By the end of this notebook, you will have cleaned five raw MIMIC-IV datasets and transformed them into a single, standardized format with four columns: `subject_id`, `date`, `code`, and `coding_system`. These cleaned files will be ready for code rollup in the next notebook.

**Recap:** In the [Getting Started](github_repo/TUTORIAL/EHR_Preprocessing_Getting_Started.ipynb) notebook, we explored the MIMIC-IV data structure and learned that every useful EHR record needs three elements: a **Patient ID** (`subject_id`), a **Clinical Code** (such as `icd_code`), and a **Timestamp** (such as `admittime`). Now we will clean and reshape these elements into a consistent format across all five datasets.

> **Full executable notebook:** [github_repo/TUTORIAL/EHR_Preprocessing_1_Data_Cleaning.ipynb](github_repo/TUTORIAL/EHR_Preprocessing_1_Data_Cleaning.ipynb)

## 1. Setup

First, let's import the libraries we need and point to our workspace. Update `base_dir` to match your `EHR_TUTORIAL_WORKSPACE` location.

In [ ]:
import os
import pandas as pd
from tqdm import tqdm

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# Update this path to your EHR_TUTORIAL_WORKSPACE location
base_dir = "EHR_TUTORIAL_WORKSPACE"

We also need an output directory for our cleaned files. All batch files across every data type will be saved here.

In [ ]:
cleaned_data_dir = os.path.join(base_dir, 'processed_data', 'step3_cleaned_rawdata')
os.makedirs(cleaned_data_dir, exist_ok=True)

print(f"Output directory: {cleaned_data_dir}")

We will also define a shortcut to the MIMIC-IV hospital module directory, since every raw file lives there.

In [ ]:
hosp_dir = os.path.join(
    base_dir, "raw_data", "structured_data",
    "physionet.org", "files", "mimiciv", "3.1", "hosp"
)

## 2. What Does Data Cleaning Involve?

Each raw MIMIC-IV file has different column names, different timestamp formats, and may contain missing or duplicate records. Our job is to bring every dataset into a common shape so that downstream analysis can treat them uniformly.

We will apply four cleaning steps to each dataset:

1. **Assess Missingness** -- Check whether any critical columns (patient ID, code, date) have missing values. If the proportion is small, drop those rows. If large, investigate further.
2. **Filter and Merge** -- Keep only the columns we need and, when necessary, merge tables to attach timestamps.
3. **Standardize Schema** -- Rename columns to a common format (`subject_id`, `date`, `code`, `coding_system`) and truncate timestamps to dates.
4. **Remove Duplicates** -- Drop identical rows to prevent double-counting.

We will walk through each step on the **diagnoses** data first. Once the logic is clear, we will wrap it into a reusable function and apply it to the remaining four datasets.

> **Note on erroneous data:** In real-world EHR datasets, there is an additional step -- filtering out records with impossible dates or lab values. MIMIC-IV does not require this because its dates are shifted for de-identification. We cover this topic in a note at the end of the notebook.

## 3. Step-by-Step Walkthrough: Cleaning Diagnoses

### Loading the Data

Diagnoses are unique among our datasets: the diagnosis codes live in `diagnoses_icd.csv`, but the timestamps live in a separate table called `admissions.csv`. We need both files to build a complete record.

In [ ]:
diagnoses_icd = pd.read_csv(os.path.join(hosp_dir, "diagnoses_icd.csv"), dtype=str)
admissions = pd.read_csv(os.path.join(hosp_dir, "admissions.csv"), dtype=str)

print(f"Diagnoses: {diagnoses_icd.shape[0]} rows, columns: {list(diagnoses_icd.columns)}")
print(f"Admissions: {admissions.shape[0]} rows, columns: {list(admissions.columns)}")

The diagnoses table gives us the patient ID (`subject_id`), hospital admission ID (`hadm_id`), diagnosis code (`icd_code`), and code version (`icd_version`). The admissions table gives us the admission timestamp (`admittime`). We will connect them using `subject_id` and `hadm_id`.

### Assessing Missingness

Before combining the tables, we check for missing values. We do this because a record without a patient ID, code, or date cannot be used in any downstream analysis, so we need to know if any exist.

In [ ]:
print("Diagnoses - Missing Values")
print(diagnoses_icd.isna().sum())

In [ ]:
print("Admissions - Missing Values")
print(admissions.isna().sum())

The columns we need (`subject_id`, `hadm_id`, `icd_code`, `admittime`) all have zero missing values. Some other admissions columns like `deathtime` have missing values, but we will not use those. Since our key columns are complete, we can proceed to the merge.

### Filtering Columns and Merging

We only need a handful of columns from each table. Filtering early keeps memory usage low, which matters when working with larger datasets like labs.

In [ ]:
print(f"Diagnoses shape before merge: {diagnoses_icd.shape}")

timed_diagnoses = pd.merge(
    diagnoses_icd[["subject_id", "hadm_id", "icd_code", "icd_version"]],
    admissions[["subject_id", "hadm_id", "admittime"]],
    how="left",
    on=["subject_id", "hadm_id"],
)

print(f"Diagnoses shape after merge: {timed_diagnoses.shape}")

The row count stayed the same before and after the merge. This tells us that every diagnosis matched exactly one admission -- which makes sense, because diagnoses are recorded per hospital visit.

After any merge, it is good practice to check for newly introduced missing values. A mismatch in data types (for example, `subject_id` stored as an integer in one table and a string in another) can cause joins to silently fail, leaving null values in the merged columns.

In [ ]:
print(f"Shape before dropping nulls: {timed_diagnoses.shape}")

timed_diagnoses = timed_diagnoses.dropna(
    subset=["subject_id", "icd_code", "icd_version", "admittime"]
)

print(f"Shape after dropping nulls:  {timed_diagnoses.shape}")

No rows were lost. Every diagnosis record has a valid patient ID, code, and date. If rows *had* been dropped here, it would signal a data quality issue worth investigating before continuing.

### Standardizing the Schema

Now we reshape the data into the standard four-column format that all datasets will share. This happens in three small steps.

**Step 1: Truncate timestamps to dates.** We only need day-level precision for this tutorial. If you were working with time-series data (such as ECG signals), you would keep the full timestamp.


In [ ]:
timed_diagnoses["admittime"] = timed_diagnoses["admittime"].str[:10]

display(timed_diagnoses.head(3))

Notice the `admittime` column now shows just the date portion (e.g., `2180-05-06` instead of `2180-05-06 22:23:00`).

**Step 2: Add a `coding_system` label.** We do this because the same condition can have different codes under ICD-9 versus ICD-10. The downstream code rollup notebook needs to know which mapping table to use for each row.

In [ ]:
timed_diagnoses["coding_system"] = "ICD" + timed_diagnoses["icd_version"]

display(timed_diagnoses[["icd_version", "coding_system"]].drop_duplicates())

This tells us the dataset contains both ICD9 and ICD10 codes. Each row is now labeled so we know which version it belongs to.

**Step 3: Rename and select the final four columns.** We rename `admittime` to `date` and `icd_code` to `code` so that every dataset uses the same column names.

In [ ]:
timed_diagnoses = timed_diagnoses.rename(columns={"admittime": "date", "icd_code": "code"})
timed_diagnoses = timed_diagnoses[["subject_id", "date", "code", "coding_system"]]

display(timed_diagnoses.head())

The data now has exactly the four columns we want. Every dataset we clean will end up looking like this.

### Removing Duplicates

Duplicate records are common in EHR data. **Exact duplicates** are identical rows that got recorded more than once. **Semantic duplicates** occur when different codes represent the same clinical concept -- we will handle those in the Code Rollup notebook. For now, we remove exact duplicates only.

In [ ]:
print(f"Shape before deduplication: {timed_diagnoses.shape}")

timed_diagnoses = timed_diagnoses.drop_duplicates()

print(f"Shape after deduplication:  {timed_diagnoses.shape}")

No duplicates were found in the diagnoses data. Other datasets -- especially labs, where the same test may be recorded multiple times on the same day -- are more likely to contain duplicates.

---

That completes the walkthrough. We assessed missingness, filtered and merged columns, standardized the schema, and removed duplicates. Every step followed the same rhythm: *look at the data, transform it, verify the result.* Next, we will scale this up to handle datasets too large to fit in memory.

## 4. Why Batch Processing?

The cleaning steps above worked well for diagnoses (~6 million rows), but the lab dataset has over **158 million rows**. Loading that into memory at once would crash most machines.

The solution is **batch processing**: we split patients into groups and process one group at a time. We batch at the patient level (not the row level) so that all records for a given patient stay together. This matters because deduplication needs to see all of a patient's records to work correctly.

Let's start by identifying all unique patients and assigning each one to a batch.

In [ ]:
admissions = pd.read_csv(os.path.join(hosp_dir, "admissions.csv"), dtype=str)

patient_batches = admissions[["subject_id"]].drop_duplicates()
patient_batches = patient_batches.sort_values("subject_id").reset_index(drop=True)

print(f"Total unique patients: {len(patient_batches)}")

Now we assign each patient to one of 8 batches using round-robin assignment. You can increase `num_batches` if you have limited memory, or decrease it if you have plenty.

In [ ]:
# Increase this number if you run into memory issues
num_batches = 8

patient_batches["batch_num"] = (patient_batches.index % num_batches) + 1

print("Patients per batch:")
print(patient_batches.groupby("batch_num")["subject_id"].count().to_string())

Each batch contains roughly 28,000 patients. The round-robin assignment (patient 1 goes to batch 1, patient 2 to batch 2, ..., patient 9 back to batch 1) ensures the batches are nearly equal in size.

## 5. Building the Reusable Cleaning Function

Now we wrap our walkthrough steps into a single reusable function. The function follows the exact same sequence you saw above:

1. Read the CSV in chunks (so large files like labs do not overwhelm memory)
2. For each batch, filter chunks to that batch's patients
3. Drop rows with missing values
4. Truncate timestamps to dates
5. Rename columns to the standard schema and add a coding system label
6. Remove duplicates
7. Save the cleaned batch to a CSV

Instead of using a dictionary to pass column names (which can be hard to read), the function uses **named parameters** like `code_col="icd_code"` and `date_col="admittime"` so that each argument is self-explanatory.

> **Note on function length:** This function is longer than our usual code cells because it encapsulates the entire cleaning pipeline in one place. The `# --- Collect ---`, `# --- Clean ---`, and `# --- Save ---` markers divide it into three phases that map directly to the walkthrough steps above.

In [ ]:
def clean_data_by_batch(input_file, output_dir, patient_batches,
                        code_col, date_col, coding_system_label,
                        data_name, code_version_col=None,
                        chunk_size=15_000_000):
    """
    Clean an EHR dataset in patient-level batches.

    Parameters:
        input_file:           path to the raw CSV
        output_dir:           where to save cleaned batch files
        patient_batches:      DataFrame with subject_id and batch_num
        code_col:             column name for the clinical code (e.g. "icd_code")
        date_col:             column name for the timestamp (e.g. "admittime")
        coding_system_label:  label for the coding system (e.g. "ICD")
        data_name:            output folder name (e.g. "Diagnoses")
        code_version_col:     optional version column (e.g. "icd_version");
                              if provided, coding_system = label + version (e.g. "ICD9")
        chunk_size:           rows per chunk when reading the CSV
    """
    source_name = os.path.splitext(os.path.basename(input_file))[0]
    data_output_dir = os.path.join(output_dir, data_name)
    os.makedirs(data_output_dir, exist_ok=True)

    # Only read the columns we need to keep memory usage low
    usecols = ["subject_id", date_col, code_col]
    if code_version_col:
        usecols.append(code_version_col)

    unique_batches = sorted(patient_batches["batch_num"].unique())

    for batch_num in tqdm(unique_batches, desc=f"Cleaning {data_name}"):

        # --- Collect: gather rows for this batch across all file chunks ---
        batch_patient_ids = patient_batches[
            patient_batches["batch_num"] == batch_num
        ][["subject_id"]]
        collected = []

        for chunk in pd.read_csv(input_file, chunksize=chunk_size,
                                 usecols=usecols, dtype=str):
            matched = chunk.merge(batch_patient_ids, on="subject_id",
                                  how="inner")
            if not matched.empty:
                collected.append(matched)

        if not collected:
            print(f"  Warning: No data found for batch {batch_num}")
            continue

        batch_df = pd.concat(collected, ignore_index=True)
        rows_loaded = len(batch_df)

        # --- Clean: same steps as the walkthrough ---
        batch_df = batch_df.dropna(subset=["subject_id", date_col, code_col])
        rows_after_dropna = len(batch_df)

        batch_df[date_col] = batch_df[date_col].str[:10]
        batch_df = batch_df.rename(columns={date_col: "date",
                                            code_col: "code"})

        if code_version_col:
            batch_df["coding_system"] = (coding_system_label
                                         + batch_df[code_version_col])
        else:
            batch_df["coding_system"] = coding_system_label

        batch_df = batch_df[["subject_id", "date", "code", "coding_system"]]
        batch_df = batch_df.drop_duplicates()

        # --- Save: one CSV per batch for parallel downstream processing ---
        output_file = os.path.join(
            data_output_dir,
            f"{data_name.lower()}_batch{batch_num}_{source_name}.csv"
        )
        batch_df.to_csv(output_file, index=False)

        nulls_dropped = rows_loaded - rows_after_dropna
        dupes_dropped = rows_after_dropna - len(batch_df)
        print(f"  Batch {batch_num}: {rows_loaded} rows loaded"
              + (f", {nulls_dropped} nulls dropped" if nulls_dropped else "")
              + (f", {dupes_dropped} duplicates removed" if dupes_dropped else "")
              + f" -> {len(batch_df)} rows saved")

    print(f"Done! All batches saved to: {data_output_dir}")

Each section of the function maps directly to a step from the walkthrough. The `# --- Collect ---`, `# --- Clean ---`, and `# --- Save ---` markers show where each phase begins. For every batch, the function reports how many rows were loaded, how many nulls were dropped, how many duplicates were removed, and how many rows were saved -- so you can verify the cleaning at a glance.

Now let's apply this function to all five datasets.

## 6. Cleaning Diagnoses

Diagnoses is the only dataset where the codes and timestamps live in separate files. We need to merge them first and save the result to a temporary file. We do this because our function reads from disk in chunks -- a consistent file-based interface that is essential for the 158-million-row labs file.

In [ ]:
diagnoses_icd = pd.read_csv(os.path.join(hosp_dir, "diagnoses_icd.csv"), dtype=str)
admissions = pd.read_csv(os.path.join(hosp_dir, "admissions.csv"), dtype=str)

print(f"Diagnoses: {diagnoses_icd.shape}")
print(f"Admissions: {admissions.shape}")

Now we merge the two tables to attach the admission date to each diagnosis.

In [ ]:
print(f"Diagnoses shape before merge: {diagnoses_icd.shape}")

timed_diagnoses = pd.merge(
    diagnoses_icd,
    admissions[["subject_id", "hadm_id", "admittime"]],
    how="left",
    on=["subject_id", "hadm_id"],
)

print(f"Diagnoses shape after merge:  {timed_diagnoses.shape}")

Row count unchanged -- every diagnosis matched an admission, just as we saw in the walkthrough. Now we save this merged table to a temporary CSV so the batch function can read it in chunks.

In [ ]:
tmp_dir = os.path.join(base_dir, "scripts", "tmp")
os.makedirs(tmp_dir, exist_ok=True)

timed_diagnoses_file = os.path.join(tmp_dir, "timed_diagnoses_icd.csv")
timed_diagnoses.to_csv(timed_diagnoses_file, index=False)

print(f"Saved merged diagnoses to: {timed_diagnoses_file}")

In [ ]:
clean_data_by_batch(
    input_file=timed_diagnoses_file,
    output_dir=cleaned_data_dir,
    patient_batches=patient_batches,
    code_col="icd_code",
    date_col="admittime",
    coding_system_label="ICD",
    data_name="Diagnoses",
    code_version_col="icd_version",
)

Each batch reports how many rows were saved (and how many duplicates were removed, if any). The cleaned diagnosis files now live in the `Diagnoses/` output folder.

## 7. Cleaning Procedures

Procedure data in MIMIC-IV comes from two separate sources, each using a different coding system:

- **`hcpcsevents.csv`** uses HCPCS codes (Healthcare Common Procedure Coding System), which include CPT billing codes for medical services.
- **`procedures_icd.csv`** uses ICD procedure codes (both ICD-9 and ICD-10 versions).

We clean each source separately and save both to the same `Procedures/` folder. Unlike diagnoses, both files already include a date column (`chartdate`), so no pre-merge is needed.

### HCPCS Procedures

Let's start by looking at a sample of the HCPCS data to understand its structure.

In [ ]:
hcpcs_file = os.path.join(hosp_dir, "hcpcsevents.csv")

display(pd.read_csv(hcpcs_file, nrows=5, dtype=str))

The code column is `hcpcs_cd` and the date column is `chartdate`. Since HCPCS does not have version numbers, every row will get the same coding system label: `HCPCS`.

In [ ]:
clean_data_by_batch(
    input_file=hcpcs_file,
    output_dir=cleaned_data_dir,
    patient_batches=patient_batches,
    code_col="hcpcs_cd",
    date_col="chartdate",
    coding_system_label="HCPCS",
    data_name="Procedures",
)

HCPCS is a relatively small dataset, so each batch processes quickly. Notice that all rows get the same `coding_system` label (`HCPCS`) since there are no version distinctions.

### ICD Procedures

ICD procedure codes work like diagnosis codes -- they have a version column (`icd_version`) that distinguishes ICD-9 from ICD-10. The coding system label will be `ICDPROC9` or `ICDPROC10`.

In [ ]:
procedures_icd_file = os.path.join(hosp_dir, "procedures_icd.csv")

display(pd.read_csv(procedures_icd_file, nrows=5, dtype=str))

In [ ]:
clean_data_by_batch(
    input_file=procedures_icd_file,
    output_dir=cleaned_data_dir,
    patient_batches=patient_batches,
    code_col="icd_code",
    date_col="chartdate",
    coding_system_label="ICDPROC",
    data_name="Procedures",
    code_version_col="icd_version",
)

Both HCPCS and ICD procedure files are now saved to the same `Procedures/` folder. Each batch file includes the source name in its filename (e.g., `procedures_batch1_hcpcsevents.csv` vs. `procedures_batch1_procedures_icd.csv`), so you can always tell which coding system a file contains.

## 8. Cleaning Medications

Prescriptions use **NDC** (National Drug Code) as their identifier -- a unique 10-digit code assigned to each medication product in the United States. The timestamp column is `starttime`, which records when the prescription was ordered.

In [ ]:
prescriptions_file = os.path.join(hosp_dir, "prescriptions.csv")

display(pd.read_csv(prescriptions_file, nrows=5, dtype=str))

Notice the many columns available (drug name, dose, route, etc.). For this tutorial, we only need `subject_id`, `starttime`, and `ndc`. The cleaning function will read only these columns and discard the rest.

In [ ]:
clean_data_by_batch(
    input_file=prescriptions_file,
    output_dir=cleaned_data_dir,
    patient_batches=patient_batches,
    code_col="ndc",
    date_col="starttime",
    coding_system_label="NDC",
    data_name="Medication",
)

Medications tend to have more duplicates than diagnoses because the same drug can be prescribed multiple times on the same day. The function reports how many duplicates were removed in each batch.

## 9. Cleaning Labs

Laboratory data is the **largest** dataset in MIMIC-IV -- over 158 million rows. This is where batch processing truly matters. The cleaning function will read 15 million rows at a time and filter each chunk to the current batch's patients.

Labs use `itemid` as the code (a MIMIC-specific numeric identifier for each lab test, such as `50931` for glucose) and `charttime` as the timestamp.

> **Expect this step to take approximately 30 minutes.** If it runs too slowly, try reducing `chunk_size`. If you have access to a computing cluster, this is a good candidate for parallelization (see the Advanced Module for SLURM templates).

In [ ]:
labevents_file = os.path.join(hosp_dir, "labevents.csv")

display(pd.read_csv(labevents_file, nrows=5, dtype=str))

The lab table has many columns including `value`, `valueuom` (unit of measure), and `ref_range_lower`/`ref_range_upper`. We only need `subject_id`, `charttime`, and `itemid` for this tutorial. If your research requires lab values, you would keep the additional columns and add unit-normalization logic.

In [ ]:
clean_data_by_batch(
    input_file=labevents_file,
    output_dir=cleaned_data_dir,
    patient_batches=patient_batches,
    code_col="itemid",
    date_col="charttime",
    coding_system_label="ITEMID",
    data_name="Labs",
)

Labs is the slowest dataset to clean because every batch must scan through all 158 million rows to collect its patients. If you see significantly fewer rows than expected, check that `patient_batches` was built from the same admissions file -- a mismatch in patient IDs would cause the merge to find no matches.

## 10. Validating the Output

Before moving on, let's verify that every expected batch file was created and that the output schema is consistent. This is a simple but important sanity check -- it catches issues like a batch that silently failed or a column that was accidentally dropped.

In [ ]:
expected_folders = ["Diagnoses", "Procedures", "Medication", "Labs"]

for folder in expected_folders:
    folder_path = os.path.join(cleaned_data_dir, folder)
    files = sorted(os.listdir(folder_path))
    print(f"{folder}: {len(files)} files")

Diagnoses, Medication, and Labs each have 8 files (one per batch). Procedures has 16 files because it combines two sources (HCPCS and ICD), with 8 batches each.

Now let's spot-check one file from each data type to confirm the columns are correct.

In [ ]:
expected_columns = ["subject_id", "date", "code", "coding_system"]

for folder in expected_folders:
    folder_path = os.path.join(cleaned_data_dir, folder)
    first_file = sorted(os.listdir(folder_path))[0]
    sample = pd.read_csv(os.path.join(folder_path, first_file), dtype=str, nrows=3)

    schema_ok = list(sample.columns) == expected_columns
    print(f"\n{folder} -- {first_file}")
    print(f"  Columns match expected schema: {schema_ok}")
    display(sample)

All data types have the expected four-column schema. The output directory structure looks like this:

```
EHR_TUTORIAL_WORKSPACE/
└── processed_data/
    └── step3_cleaned_rawdata/
        ├── Diagnoses/           (8 batch files)
        ├── Procedures/          (16 batch files: 8 HCPCS + 8 ICD)
        ├── Medication/          (8 batch files)
        └── Labs/                (8 batch files)
```

## 11. What We Accomplished

In this notebook, we:

1. **Walked through four cleaning steps** on the diagnoses data: assessing missingness, filtering and merging columns, standardizing the schema, and removing duplicates.
2. **Learned why batch processing is necessary** for large EHR datasets and how to split patients into batches.
3. **Built a single reusable function** (`clean_data_by_batch`) that applies the same cleaning logic to any dataset.
4. **Cleaned all five MIMIC-IV datasets** into a standardized four-column format.
5. **Validated** that the output files exist and have the correct schema.

### A Note on Erroneous Data

In real-world (non-MIMIC) EHR datasets, you should also filter out **erroneous records** after standardizing the schema:

- **Invalid dates:** Records with dates before a reasonable start year (e.g., 1980) or after the current year usually indicate data entry errors and should be removed.
- **Impossible lab values:** A negative hemoglobin level or a blood pressure of 9,000 mmHg are clearly errors that can be safely dropped.
- **Unit mismatches:** The same lab test may be recorded in different units across institutions (e.g., hemoglobin in g/dL vs. mmol/L). Normalizing units requires domain knowledge but is essential for consistent analysis.

We skip these checks for MIMIC-IV because the dates have been shifted for de-identification, and we do not use lab values in this tutorial.

---

**Next:** In the [Code Rollup notebook](github_repo/TUTORIAL/EHR_Preprocessing_2_Code_Rollup.ipynb), we will map these granular codes to standardized parent codes -- reducing thousands of individual codes into clinically meaningful groups.

> **Full executable notebook:** [github_repo/TUTORIAL/EHR_Preprocessing_1_Data_Cleaning.ipynb](github_repo/TUTORIAL/EHR_Preprocessing_1_Data_Cleaning.ipynb)